### Testing several permutations of PCA processing of dataset with GridSearchCV

In [19]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb 
from datetime import datetime as dt

In [20]:
pca_file_names = ["pca_transformed_data_0.5_target_variance.csv", "pca_transformed_data_0.6_target_variance.csv", "pca_transformed_data_0.7_target_variance.csv", "pca_transformed_data_0.8_target_variance.csv", "pca_transformed_data_0.9_target_variance.csv", "pca_transformed_data_0.95_target_variance.csv"]

pca_datasets = []

for i in pca_file_names:
    df = pd.read_csv(rf'../dataset/{i}')
    pca_datasets.append(df)

param_grid = {
    # Feature Subsampling: Forces trees to look beyond PC1-PC3
    'colsample_bytree': [0.5, 0.7, 0.9],
    
    # Tree Depth: Lower depths prevent fitting to minor noise in late PCs (e.g., PC50-PC71)
    'max_depth': [3, 5, 7],
    
    # Regularization: Critical to suppress noise in lower-variance components
    'gamma': [0, 0.2, 0.5],
    'reg_lambda': [1.0, 5.0, 10.0],
    'min_child_weight': [1, 3, 5],
    
    # Boosting Pace & Iterations
    'n_estimators': [100, 250],
    'learning_rate': [0.03, 0.1],
    'subsample': [0.8]
}

In [21]:
xgb = xgb.XGBClassifier(
    objective = 'multi:softprob',
    eval_metric = 'mlogloss',
    tree_method = 'hist',
    random_state = '69',
    n_jobs = -1
)

gkf = GroupKFold(n_splits=5)

grid_search = GridSearchCV(
    estimator = xgb,
    param_grid = param_grid,
    scoring = 'f1_macro',
    cv = gkf,
    verbose = 1,
    n_jobs = -1,
)

In [22]:
# 2D array with train, test data for each PCA permutation:
pca_dataset_splits = [
    [],
    []
]

best_parameters = [
    [],
    []
]

best_parameter_df = pd.DataFrame(columns=["dataset_name", "parameters"])

# Fix: Enumerate over datasets to match the correct pca_file_names index
for idx, j in enumerate(pca_datasets):
    column_count = j.columns.tolist()
    # Number of PC columns = Total columns minus 'subject' and 'Activity'
    pca_component_count = len(column_count) - 2
    
    # Fix: Include the last component by using + 1
    X = j[[f"PC{i}" for i in range(1, pca_component_count + 1)]]
    
    # Fix: Encode string labels ('LAYING', etc.) to 0, 1, 2, 3, 4, 5
    le = LabelEncoder()
    y = le.fit_transform(j['Activity'])
    
    groups = j['subject']

    gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=69)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    groups_train = groups.iloc[train_idx]
    groups_test = groups.iloc[test_idx]

    grid_search.fit(X_train, y_train, groups=groups_train)

    # Fix: Use idx to log the correct file name
    best_parameter_df.loc[len(best_parameter_df)] = [pca_file_names[idx], grid_search.best_params_]

now_str = dt.now().strftime("%Y-%m-%d_%H-%M-%S")


best_parameter_df.to_csv(rf'../dataset/best_parameter_for_pca_permutations_{now_str}.csv', index=False)



Fitting 5 folds for each of 972 candidates, totalling 4860 fits


Fitting 5 folds for each of 972 candidates, totalling 4860 fits
Fitting 5 folds for each of 972 candidates, totalling 4860 fits
Fitting 5 folds for each of 972 candidates, totalling 4860 fits
Fitting 5 folds for each of 972 candidates, totalling 4860 fits


/Users/jonaskarlsen/Documents/git/haml-et/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Fitting 5 folds for each of 972 candidates, totalling 4860 fits


/Users/jonaskarlsen/Documents/git/haml-et/.venv/lib/python3.13/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
